# OULAD Data Quality: CLEAN layer

Source tables: `` `ftw-week-07`.`02-clean` ``  
Results table: `` `ftw-week-07`.`01-raw`.dq_check_results ``

Every check declares WHAT, EXPECTATION, THRESHOLD, SEVERITY, OWNER. 

| Status | Meaning | Action |
|---|---|---|
| `PASS` | Expectation held | None |
| `WARN` | Known issue, quantified and documented | Review, don't block |
| `FAIL` | Expectation broken | **Stop. Do not build Mart.** |
| `INFO` | A measurement, not a test | Excluded from pass rate |

| Severity | Meaning |
|---|---|
| `FAIL` | Data must not continue to Mart layer |
| `WARN` | Issue should be reviewed but won't block pipeline |

**Clean layer focus:**
* Validate transformations worked correctly
* Verify data quality flags are set appropriately
* Ensure no new issues introduced during cleaning
* Reconcile with raw layer (row counts match or explained by deduplication)
* Check that NULL conversions and type casts succeeded

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 1. RUN CONTEXT
--    Variable is dq_run_id; the column is run_id. Different names on
--    purpose — an unqualified reference to a name that matches a column
--    resolves to the column and silently breaks the filter.
-- ---------------------------------------------------------------------
DECLARE OR REPLACE VARIABLE dq_run_id STRING;
SET VARIABLE dq_run_id = uuid();

In [0]:
%sql
-- ========== courses ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'courses';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.courses
),
checks AS (
    SELECT 'unique_module_presentation' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: (code_module, code_presentation) must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT code_module, code_presentation, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.courses
               GROUP BY code_module, code_presentation HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_code_module', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'code_module is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.courses WHERE code_module IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_code_presentation', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'code_presentation is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.courses WHERE code_presentation IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_module_presentation_length', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'module_presentation_length required after type cast',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.courses WHERE module_presentation_length IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_module_presentation_length', 'RANGE', 'FAIL', 0.0, 'data-engineering',
           'Presentation length must be positive',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.courses WHERE module_presentation_length <= 0),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'courses',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE 'FAIL' END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== assessments ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'assessments';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.assessments
),
checks AS (
    SELECT 'unique_id_assessment' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: id_assessment must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT id_assessment, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.assessments
               GROUP BY id_assessment HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_id_assessment', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_assessment is required PK',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments WHERE id_assessment IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_assessment_type', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'assessment_type required',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments WHERE assessment_type IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'domain_assessment_type', 'DOMAIN', 'FAIL', 0.0, 'data-engineering',
           'assessment_type must be TMA, CMA, or Exam',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments 
            WHERE assessment_type NOT IN ('TMA', 'CMA', 'Exam')),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_weight', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'weight required after type cast',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments WHERE weight IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_weight', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'Weight should be between 0 and 100',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments 
            WHERE weight < 0 OR weight > 100),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'flag_has_missing_due_date', 'CONSISTENCY', 'FAIL', 0.0, 'data-engineering',
           'has_missing_due_date flag should match due_day_offset NULL status',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments 
            WHERE (due_day_offset IS NULL AND has_missing_due_date = FALSE) 
               OR (due_day_offset IS NOT NULL AND has_missing_due_date = TRUE)),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_missing_due_date', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Assessments with no due date. Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments WHERE has_missing_due_date = TRUE),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_courses', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All assessments must reference valid courses',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.assessments a
            LEFT JOIN `ftw-week-07`.`02-clean`.courses c
              ON a.code_module = c.code_module 
             AND a.code_presentation = c.code_presentation
            WHERE c.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'assessments',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== vle ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'vle';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.vle
),
checks AS (
    SELECT 'unique_id_site' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: id_site must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT id_site, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.vle
               GROUP BY id_site HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_id_site', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_site is required PK',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.vle WHERE id_site IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_activity_type', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'activity_type required',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.vle WHERE activity_type IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'flag_has_missing_week_boundaries', 'CONSISTENCY', 'FAIL', 0.0, 'data-engineering',
           'has_missing_week_boundaries flag should match week_from/week_to NULL status',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.vle 
            WHERE (week_from IS NULL OR week_to IS NULL) AND has_missing_week_boundaries = FALSE
               OR (week_from IS NOT NULL AND week_to IS NOT NULL AND has_missing_week_boundaries = TRUE)),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_no_week_range', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Resources with no week range. Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.vle WHERE has_missing_week_boundaries = TRUE),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_courses', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All VLE resources must reference valid courses',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.vle v
            LEFT JOIN `ftw-week-07`.`02-clean`.courses c
              ON v.code_module = c.code_module 
             AND v.code_presentation = c.code_presentation
            WHERE c.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'vle',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== student_info ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'student_info';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.student_info
),
checks AS (
    SELECT 'unique_student_enrollment' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: (code_module, code_presentation, id_student) must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT code_module, code_presentation, id_student, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.student_info
               GROUP BY code_module, code_presentation, id_student HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_id_student', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_student is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info WHERE id_student IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_final_result', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'final_result required',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info WHERE final_result IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'domain_final_result', 'DOMAIN', 'FAIL', 0.0, 'data-engineering',
           'final_result must be Pass, Fail, Distinction, or Withdrawn',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info 
            WHERE final_result NOT IN ('Pass', 'Fail', 'Distinction', 'Withdrawn')),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'flag_has_missing_imd_band', 'CONSISTENCY', 'FAIL', 0.0, 'data-engineering',
           'has_missing_imd_band flag should match imd_band NULL status',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info 
            WHERE (imd_band IS NULL AND has_missing_imd_band = FALSE) 
               OR (imd_band IS NOT NULL AND has_missing_imd_band = TRUE)),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_missing_imd_band', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Enrollments with no IMD band. Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info WHERE has_missing_imd_band = TRUE),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_num_of_prev_attempts', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'num_of_prev_attempts should be non-negative',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info WHERE num_of_prev_attempts < 0),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_studied_credits', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'studied_credits should be positive',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info WHERE studied_credits <= 0),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_distinct_students', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Distinct students vs enrollments. Reconciles to Raw.',
           (SELECT COUNT(DISTINCT id_student) FROM `ftw-week-07`.`02-clean`.student_info),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_courses', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All student enrollments must reference valid courses',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_info si
            LEFT JOIN `ftw-week-07`.`02-clean`.courses c
              ON si.code_module = c.code_module 
             AND si.code_presentation = c.code_presentation
            WHERE c.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'student_info',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== student_registration ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'student_registration';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.student_registration
),
checks AS (
    SELECT 'unique_student_enrollment' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: (code_module, code_presentation, id_student) must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT code_module, code_presentation, id_student, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.student_registration
               GROUP BY code_module, code_presentation, id_student HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_id_student', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_student is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration WHERE id_student IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_date_registration', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'date_registration required after type cast',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration WHERE date_registration IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'flag_completed_course', 'CONSISTENCY', 'FAIL', 0.0, 'data-engineering',
           'completed_course flag should match date_unregistration NULL status',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration 
            WHERE (date_unregistration IS NULL AND completed_course = FALSE) 
               OR (date_unregistration IS NOT NULL AND completed_course = TRUE)),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_completed_course', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Students who completed course (never withdrew). Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration WHERE completed_course = TRUE),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_date_registration', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'Registration date should be reasonable (within presentation window)',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration 
            WHERE date_registration < -30 OR date_registration > 400),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_date_unregistration', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'Unregistration date should be after registration',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration 
            WHERE date_unregistration IS NOT NULL AND date_unregistration < date_registration),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_student_info', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All registrations must have matching student_info record',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_registration sr
            LEFT JOIN `ftw-week-07`.`02-clean`.student_info si
              ON sr.code_module = si.code_module 
             AND sr.code_presentation = si.code_presentation
             AND sr.id_student = si.id_student
            WHERE si.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'student_registration',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== student_assessment ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'student_assessment';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.student_assessment
),
checks AS (
    SELECT 'unique_submission' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: (id_assessment, id_student) must be unique' AS details,
           (SELECT COUNT(*) FROM (
               SELECT id_assessment, id_student, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.student_assessment
               GROUP BY id_assessment, id_student HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_id_assessment', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_assessment is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment WHERE id_assessment IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_id_student', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'id_student is required PK component',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment WHERE id_student IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_score', 'RANGE', 'WARN', 0.01, 'data-engineering',
           'Score should be between 0 and 100 when present',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment 
            WHERE score IS NOT NULL AND (score < 0 OR score > 100)),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_missing_score', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Submissions with no score. Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment WHERE score IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_banked_scores', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Scores transferred from earlier presentation. Reconciles to Raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment WHERE is_banked = 1),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_assessments', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All submissions must reference valid assessments',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_assessment sa
            LEFT JOIN `ftw-week-07`.`02-clean`.assessments a
              ON sa.id_assessment = a.id_assessment
            WHERE a.id_assessment IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_student_info', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All submissions must have matching student enrollment',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_assessment sa
            JOIN `ftw-week-07`.`02-clean`.assessments a
              ON sa.id_assessment = a.id_assessment
            LEFT JOIN `ftw-week-07`.`02-clean`.student_info si
              ON a.code_module = si.code_module 
             AND a.code_presentation = si.code_presentation
             AND sa.id_student = si.id_student
            WHERE si.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'student_assessment',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== student_vle ==========
-- Large table: use conditional aggregation to fold row-level checks into one pass
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND dataset = 'student_vle';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`02-clean`.student_vle
),
row_checks AS (
    SELECT 
        SUM(CASE WHEN code_module IS NULL OR code_presentation IS NULL 
                      OR id_student IS NULL OR id_site IS NULL OR date IS NULL THEN 1 ELSE 0 END) AS null_pk,
        SUM(CASE WHEN sum_click IS NULL THEN 1 ELSE 0 END) AS null_sum_click,
        SUM(CASE WHEN sum_click < 0 THEN 1 ELSE 0 END) AS negative_clicks
    FROM `ftw-week-07`.`02-clean`.student_vle
),
checks AS (
    SELECT 'unique_student_site_date' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct, 'data-engineering' AS owner,
           'PK: (code_module, code_presentation, id_student, id_site, date) must be unique after dedup' AS details,
           (SELECT COUNT(*) FROM (
               SELECT code_module, code_presentation, id_student, id_site, date, COUNT(*) AS cnt
               FROM `ftw-week-07`.`02-clean`.student_vle
               GROUP BY code_module, code_presentation, id_student, id_site, date HAVING cnt > 1
           )) AS fail_count,
           (SELECT n FROM total) AS total_count
    
    UNION ALL
    SELECT 'not_null_pk_components', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'PK components (code_module, code_presentation, id_student, id_site, date) required',
           (SELECT null_pk FROM row_checks),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'not_null_sum_click', 'NOT_NULL', 'FAIL', 0.0, 'data-engineering',
           'sum_click required after type cast',
           (SELECT null_sum_click FROM row_checks),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'range_sum_click', 'RANGE', 'FAIL', 0.0, 'data-engineering',
           'sum_click must be non-negative',
           (SELECT negative_clicks FROM row_checks),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'measure_multiple_records', 'MEASURE', 'FAIL', 0.0, 'data-engineering',
           'Rows that had multiple records aggregated (clean < raw). Clean row count should be less than raw.',
           (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_vle WHERE had_multiple_records = TRUE),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_vle', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All student VLE interactions must reference valid VLE resources',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_vle sv
            LEFT JOIN `ftw-week-07`.`02-clean`.vle v
              ON sv.id_site = v.id_site
            WHERE v.id_site IS NULL),
           (SELECT n FROM total)
    
    UNION ALL
    SELECT 'referential_to_student_info', 'REFERENTIAL', 'FAIL', 0.0, 'data-engineering',
           'All student VLE interactions must have matching student enrollment',
           (SELECT COUNT(*)
            FROM `ftw-week-07`.`02-clean`.student_vle sv
            LEFT JOIN `ftw-week-07`.`02-clean`.student_info si
              ON sv.code_module = si.code_module 
             AND sv.code_presentation = si.code_presentation
             AND sv.id_student = si.id_student
            WHERE si.code_module IS NULL),
           (SELECT n FROM total)
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'student_vle',
       check_name, check_type,
       CASE WHEN fail_count = 0 THEN 'PASS'
            WHEN fail_count / total_count <= threshold_pct THEN 'PASS'
            ELSE severity END,
       severity, fail_count, total_count,
       CASE WHEN total_count > 0 THEN fail_count / total_count ELSE 0.0 END,
       threshold_pct, CAST(NULL AS DOUBLE), owner, details
FROM checks;

In [0]:
%sql
-- ========== cross-table consistency ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean'
  AND check_name IN ('withdrawal_flag_agrees_with_date', 'row_count_not_empty');

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n 
    FROM `ftw-week-07`.`02-clean`.student_info si
    JOIN `ftw-week-07`.`02-clean`.student_registration sr
      ON si.code_module = sr.code_module
     AND si.code_presentation = sr.code_presentation
     AND si.id_student = sr.id_student
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'cross_table',
       'withdrawal_flag_agrees_with_date' AS check_name, 'CONSISTENCY' AS check_type,
       CASE WHEN COUNT(*) = 0 THEN 'PASS' ELSE 'WARN' END AS status,
       'WARN' AS severity,
       COUNT(*) AS fail_count,
       (SELECT n FROM total) AS total_count,
       CASE WHEN (SELECT n FROM total) > 0 
            THEN CAST(COUNT(*) AS DOUBLE) / (SELECT n FROM total) 
            ELSE 0.0 END AS fail_pct,
       0.01 AS threshold_pct,
       CAST(NULL AS DOUBLE) AS metric_value,
       'data-engineering' AS owner,
       'Students marked Withdrawn should have unregistration date, and vice versa' AS details
FROM `ftw-week-07`.`02-clean`.student_info si
JOIN `ftw-week-07`.`02-clean`.student_registration sr
  ON si.code_module = sr.code_module
 AND si.code_presentation = sr.code_presentation
 AND si.id_student = sr.id_student
WHERE (si.final_result = 'Withdrawn' AND sr.completed_course = TRUE)
   OR (si.final_result <> 'Withdrawn' AND sr.completed_course = FALSE);

In [0]:
%sql
-- ========== reconciliation: clean vs raw row counts ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean'
  AND check_name LIKE 'reconcile_%';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH raw_counts AS (
    SELECT 'courses' AS dataset, COUNT(*) AS raw_n FROM `ftw-week-07`.`01-raw`.courses
    UNION ALL SELECT 'assessments', COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments
    UNION ALL SELECT 'vle', COUNT(*) FROM `ftw-week-07`.`01-raw`.vle
    UNION ALL SELECT 'student_info', COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info
    UNION ALL SELECT 'student_registration', COUNT(*) FROM `ftw-week-07`.`01-raw`.student_registration
    UNION ALL SELECT 'student_assessment', COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment
    UNION ALL SELECT 'student_vle', COUNT(*) FROM `ftw-week-07`.`01-raw`.student_vle
),
clean_counts AS (
    SELECT 'courses' AS dataset, COUNT(*) AS clean_n FROM `ftw-week-07`.`02-clean`.courses
    UNION ALL SELECT 'assessments', COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments
    UNION ALL SELECT 'vle', COUNT(*) FROM `ftw-week-07`.`02-clean`.vle
    UNION ALL SELECT 'student_info', COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info
    UNION ALL SELECT 'student_registration', COUNT(*) FROM `ftw-week-07`.`02-clean`.student_registration
    UNION ALL SELECT 'student_assessment', COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment
    UNION ALL SELECT 'student_vle', COUNT(*) FROM `ftw-week-07`.`02-clean`.student_vle
),
comparison AS (
    SELECT 
        r.dataset,
        r.raw_n,
        c.clean_n,
        -- student_vle should have FEWER rows due to deduplication
        -- All other tables should match exactly
        CASE 
            WHEN r.dataset = 'student_vle' AND c.clean_n >= r.raw_n THEN 1
            WHEN r.dataset <> 'student_vle' AND c.clean_n <> r.raw_n THEN 1
            ELSE 0
        END AS is_mismatch
    FROM raw_counts r
    JOIN clean_counts c ON r.dataset = c.dataset
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', dataset,
       'reconcile_row_count_vs_raw' AS check_name,
       'RECONCILIATION' AS check_type,
       CASE WHEN is_mismatch = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
       'FAIL' AS severity,
       is_mismatch AS fail_count,
       1 AS total_count,
       CAST(is_mismatch AS DOUBLE) AS fail_pct,
       0.0 AS threshold_pct,
       CAST(raw_n AS DOUBLE) AS metric_value,
       'data-engineering' AS owner,
       CASE 
           WHEN dataset = 'student_vle' 
           THEN CONCAT('Clean should have fewer rows than raw due to deduplication. Raw: ', raw_n, ', Clean: ', clean_n)
           ELSE CONCAT('Clean should match raw row count exactly. Raw: ', raw_n, ', Clean: ', clean_n)
       END AS details
FROM comparison;

In [0]:
%sql
-- ========== reconciliation: total clicks preserved ==========
-- Deduplication must preserve the exact click total from raw
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean'
  AND check_name = 'reconciliation_total_clicks';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH raw_total AS (
    SELECT SUM(sum_click) AS raw_clicks
    FROM `ftw-week-07`.`01-raw`.student_vle
),
clean_total AS (
    SELECT SUM(sum_click) AS clean_clicks
    FROM `ftw-week-07`.`02-clean`.student_vle
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', 'student_vle',
       'reconciliation_total_clicks' AS check_name,
       'RECONCILIATION' AS check_type,
       CASE WHEN r.raw_clicks = c.clean_clicks THEN 'PASS' ELSE 'FAIL' END AS status,
       'FAIL' AS severity,
       CASE WHEN r.raw_clicks = c.clean_clicks THEN 0 ELSE 1 END AS fail_count,
       1 AS total_count,
       CASE WHEN r.raw_clicks = c.clean_clicks THEN 0.0 ELSE 1.0 END AS fail_pct,
       0.0 AS threshold_pct,
       CAST(r.raw_clicks AS DOUBLE) AS metric_value,
       'data-engineering' AS owner,
       CONCAT('Total clicks at Raw: ', r.raw_clicks, '. Clean must reproduce this exactly after aggregating. Clean total: ', c.clean_clicks) AS details
FROM raw_total r
CROSS JOIN clean_total c;

In [0]:
%sql
-- ========== volume: not empty ==========
INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', dataset,
       'row_count_not_empty', 'VOLUME',
       CASE WHEN n = 0 THEN 'FAIL' ELSE 'PASS' END, 'FAIL',
       CASE WHEN n = 0 THEN 1 ELSE 0 END, n,
       CASE WHEN n = 0 THEN 1.0 ELSE 0.0 END, 0.0,
       CAST(NULL AS DOUBLE), 'data-engineering',
       'Zero rows is a silent failure. Every other check passes vacuously on an empty table.'
FROM (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'clean' AND check_type <> 'VOLUME'
    GROUP BY dataset
);

In [0]:
%sql
-- ========== volume: stability vs previous run ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean'
  AND check_name = 'volume_stable_vs_previous_run';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH current_counts AS (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'clean' AND check_type <> 'VOLUME'
    GROUP BY dataset
),
previous_counts AS (
    SELECT dataset, total_count AS prev_n
    FROM (
        SELECT dataset, total_count,
               DENSE_RANK() OVER (PARTITION BY dataset ORDER BY executed_at DESC) AS rnk
        FROM `ftw-week-07`.`01-raw`.dq_check_results
        WHERE layer = 'clean' AND check_type <> 'VOLUME' AND run_id <> dq_run_id
    )
    WHERE rnk = 1
    GROUP BY dataset, total_count
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'clean', c.dataset,
       'volume_stable_vs_previous_run', 'VOLUME',
       CASE WHEN p.prev_n IS NULL THEN 'INFO'
            WHEN ABS(c.n - p.prev_n) / p.prev_n <= 0.10 THEN 'PASS'
            ELSE 'WARN' END,
       'WARN',
       CASE WHEN p.prev_n IS NULL OR ABS(c.n - p.prev_n) / p.prev_n <= 0.10
            THEN 0 ELSE ABS(c.n - p.prev_n) END,
       c.n,
       CASE WHEN p.prev_n IS NULL THEN NULL ELSE ABS(c.n - p.prev_n) / p.prev_n END,
       0.10,
       CAST(p.prev_n AS DOUBLE),
       'data-engineering',
       CASE WHEN p.prev_n IS NULL
            THEN 'First run for this dataset. No baseline to compare against yet.'
            ELSE 'Row count vs previous run. metric_value holds the previous count.' END
FROM current_counts c
LEFT JOIN previous_counts p ON c.dataset = p.dataset;

## Clean Exit Gate

**Mart does not start until this returns zero rows.**

In [0]:
%sql
-- ---------------------------------------------------------------------
-- CLEAN EXIT GATE
--    Mart does not start until this returns zero rows.
-- ---------------------------------------------------------------------
SELECT dataset, check_name, check_type, fail_count, total_count,
       ROUND(fail_pct * 100, 4) AS fail_pct, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'clean' AND status = 'FAIL'
ORDER BY dataset, check_type;

## Dashboard Tiles

Data health in 10 seconds. Each tile is its own cell so it can be pinned to a Databricks dashboard individually. All tiles read `v_dq_latest`, which is the most recent run per layer.

In [0]:
%sql
-- TILE 1 — Overall health + pass rate + last checked
SELECT
    layer,
    COUNT(*)                                                  AS checks_run,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)          AS passed,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END)          AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END)          AS failed,
    ROUND(100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END), 0), 1) AS pass_rate_pct,
    CASE WHEN SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) > 0 THEN 'STOP'
         WHEN SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) > 0 THEN 'REVIEW'
         ELSE 'HEALTHY' END                                   AS overall_health,
    MAX(executed_at)                                          AS last_checked
FROM `ftw-week-07`.`01-raw`.v_dq_latest
WHERE layer = 'clean'
GROUP BY layer;

In [0]:
%sql
-- TILE 2 — Failures by dataset (where is the problem?)
SELECT layer, dataset,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
       SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
       SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS passed
FROM `ftw-week-07`.`01-raw`.v_dq_latest
WHERE layer = 'clean' AND status <> 'INFO'
GROUP BY layer, dataset
ORDER BY failed DESC, warnings DESC, dataset;

In [0]:
%sql
-- TILE 3 — Failures by check type (what kind of problem?)
SELECT check_type,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed,
       SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
       COUNT(*)                                         AS total_checks
FROM `ftw-week-07`.`01-raw`.v_dq_latest
WHERE layer = 'clean' AND status <> 'INFO'
GROUP BY check_type
ORDER BY failed DESC, warnings DESC;

In [0]:
%sql
-- TILE 4 — Open issues, worst first (what is broken, and how badly?)
SELECT layer, dataset, check_name, check_type, status,
       fail_count, total_count,
       ROUND(fail_pct * 100, 4)      AS fail_pct,
       ROUND(threshold_pct * 100, 4) AS threshold_pct,
       owner, details
FROM `ftw-week-07`.`01-raw`.v_dq_latest
WHERE layer = 'clean' AND status IN ('FAIL', 'WARN')
ORDER BY CASE status WHEN 'FAIL' THEN 0 ELSE 1 END, fail_pct DESC NULLS LAST;

In [0]:
%sql
-- TILE 5 — Measurements (numbers Mart must reconcile against)
SELECT layer, dataset, check_name, metric_value, details
FROM `ftw-week-07`.`01-raw`.v_dq_latest
WHERE layer = 'clean' AND check_type IN ('MEASURE', 'RECONCILIATION')
ORDER BY dataset, check_name;

In [0]:
%sql
-- TILE 6 — Trend: is quality getting worse?
SELECT
    run_id,
    layer,
    MIN(executed_at) AS run_at,
    SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END) AS checks_scored,
    ROUND(100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END), 0), 1) AS pass_rate_pct,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) AS failed
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE layer = 'clean'
GROUP BY run_id, layer
ORDER BY run_at DESC;